In [ ]:
import datetime as dt
import sys
from pathlib import Path

# Add the workspace root to Python path so we can import from src
workspace_root = (
    Path(__file__).parent.parent.parent
    if "__file__" in globals()
    else Path.cwd().parent.parent
)
sys.path.insert(0, str(workspace_root))

import dask
import numpy as np
import pandas as pd
import importlib
from sqlalchemy import select, create_engine
from dotenv import load_dotenv
import os
import mc_postgres_db.models as models
import src.utils.stochastic as stochastic
import matplotlib.pyplot as plt

importlib.reload(stochastic)

load_dotenv()

# Infrastructure parameters
POSTGRES_URL = os.getenv("POSTGRES_URL")
CLUSTER_TYPE = "local"
N_WORKERS = 6
N_CONCCURENT_DATABASE_CALLS = 10

# Model parameters
DISCOUNT_RATE = 0.001  # Discount rate
TRANSACTION_COST = 0.001  # Transaction cost
PVALUE_THRESHOLD = 0.01  # Only trade if p_value < 0.01 (99% confidence)
PERCENT_LOSS = 0.05
CASH_ALLOCATION = 1000

dask.config.set({"distributed.scheduler.locks.lease-timeout": "120s"})  # 2 minutes

engine = create_engine(POSTGRES_URL)

In [ ]:
def load_pairs_trading_frame_chunk(
    start: dt.datetime,
    end: dt.datetime,
    provider_asset_group_id: int,
) -> pd.DataFrame:
    # Get the postgres URL.
    postgres_url = os.getenv("POSTGRES_URL")
    engine = create_engine(postgres_url)

    # Generate timeframe using pd.date_range
    time_frame_df = pd.DataFrame({"timestamp": pd.date_range(start, end, freq="1min")})

    # Use passed-in members_chunk (already filtered)
    members_df = pd.read_sql(
        select(
            models.ProviderAssetGroupMember.provider_asset_group_id,
            models.ProviderAssetGroupMember.order,
            models.ProviderAssetGroupMember.provider_id,
            models.ProviderAssetGroupMember.from_asset_id,
            models.ProviderAssetGroupMember.to_asset_id,
        ).where(
            models.ProviderAssetGroupMember.provider_asset_group_id
            == provider_asset_group_id
        ),
        engine,
    )

    # Cross join
    full_frame_df = time_frame_df.merge(members_df, how="cross")
    full_frame_df = full_frame_df.sort_values("timestamp")

    # Get market data.
    market_df: pd.DataFrame = pd.read_sql(
        select(
            models.ProviderAssetMarket.timestamp,
            models.ProviderAssetMarket.provider_id,
            models.ProviderAssetMarket.from_asset_id,
            models.ProviderAssetMarket.to_asset_id,
            models.ProviderAssetMarket.close,
        )
        .where(
            models.ProviderAssetMarket.timestamp.between(start, end),
            models.ProviderAssetMarket.from_asset_id.in_(
                members_df["from_asset_id"].unique().tolist()
            ),
            models.ProviderAssetMarket.to_asset_id.in_(
                members_df["to_asset_id"].unique().tolist()
            ),
        )
        .order_by(models.ProviderAssetMarket.timestamp),
        engine,
        parse_dates=["timestamp"],
    )
    market_df = market_df.astype(
        {
            "provider_id": "int64",
            "from_asset_id": "int64",
            "to_asset_id": "int64",
            "close": "float64",
        }
    )

    # Merge_asof
    full_market_frame = pd.merge_asof(
        full_frame_df,
        market_df,
        on="timestamp",
        by=["provider_id", "from_asset_id", "to_asset_id"],
        direction="backward",
    )

    # Split by order and create pairs - only keep essential columns
    close_1 = full_market_frame[full_market_frame["order"] == 1][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_1"})
    close_2 = full_market_frame[full_market_frame["order"] == 2][
        ["timestamp", "provider_asset_group_id", "close"]
    ].rename(columns={"close": "close_2"})

    # Merge to create pairs - only timestamp, close_1, close_2
    pairs = pd.merge(
        close_1, close_2, on=["timestamp", "provider_asset_group_id"], how="inner"
    )

    # Keep only essential columns
    pairs = pairs[["provider_asset_group_id", "timestamp", "close_1", "close_2"]]

    # Set index to provider_asset_group_id
    pairs = pairs.set_index("provider_asset_group_id")

    return pairs

In [ ]:
provider_asset_group_id = 4108
window_days = 7
window = window_days * 24 * 60
test_days = 60
lookback_days = window_days + test_days
end_time = dt.datetime.combine(dt.datetime.today(), dt.time.min) - dt.timedelta(days=1)
start_time = end_time - dt.timedelta(days=lookback_days)
print(
    f"Running pairs trading over period {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}"
)

In [ ]:
df_init = load_pairs_trading_frame_chunk(
    start_time,
    end_time,
    provider_asset_group_id,
)

In [ ]:
df = df_init.copy()

In [ ]:
result_cointegration = stochastic.RollingCointegration(
    df["close_1"].to_numpy(), df["close_2"].to_numpy(), window=7 * 24 * 60
).fit()

In [ ]:
df["beta"] = result_cointegration.beta
df["pvalue"] = result_cointegration.pvalue

In [ ]:
result_ou = stochastic.RollingOrnsteinUhlenbeck(
    df["beta"].to_numpy(),
    df["close_1"].to_numpy(),
    df["close_2"].to_numpy(),
    window=7 * 24 * 60,
).fit()

In [ ]:
df["mu"] = result_ou.mu
df["sigma"] = result_ou.sigma
df["theta"] = result_ou.theta
df["half_life"] = result_ou.half_life

In [ ]:
df["spread"] = df["close_1"] - df["beta"] * df["close_2"]
df.loc[df["pvalue"] >= PVALUE_THRESHOLD, ["mu", "theta", "sigma"]] = np.nan

In [ ]:
exit_level = np.full(len(df), np.nan)
entry_level = np.full(len(df), np.nan)
mask = df["pvalue"].to_numpy() < PVALUE_THRESHOLD
exit_level[mask] = stochastic.OrnsteinUhlenbeck.get_optimal_exit_level(
    df["mu"].to_numpy()[mask],
    df["sigma"].to_numpy()[mask],
    df["theta"].to_numpy()[mask],
    0.01,
    TRANSACTION_COST,
    max_iter=1000,
    max_initial_shift=1000,
)
df["exit_level"] = exit_level
entry_level[mask] = stochastic.OrnsteinUhlenbeck.get_optimal_entry_level(
    df["mu"].to_numpy()[mask],
    df["sigma"].to_numpy()[mask],
    df["theta"].to_numpy()[mask],
    df["exit_level"].to_numpy()[mask],
    0.01,
    TRANSACTION_COST,
    max_iter=1000,
    max_initial_shift=1000,
)
df["entry_level"] = entry_level

In [ ]:
plt.plot(df["timestamp"], df["spread"])
plt.plot(df["timestamp"], df["theta"])
plt.plot(df["timestamp"], df["exit_level"])
plt.plot(df["timestamp"], df["entry_level"])